# View GraphRAG Entities and Relationships
This notebook loads GraphRAG artifacts and displays entities and relationships as pandas DataFrames.

In [5]:
import pandas as pd
from pathlib import Path
import json 
artifacts_dir = Path('/home/karimnazarovj/RABBIT_mysql/knowledge/mysql/graph/output/artifacts')

entities_df = pd.read_parquet(artifacts_dir / 'create_final_entities.parquet')
relationships_df = pd.read_parquet(artifacts_dir / 'create_final_relationships.parquet')

# pandas display don't cut text in cells, so we can see full descriptions
# pd.set_option('display.max_colwidth', None)

# load up all the names of knobs from knowledge/postgres/system_view_origin.json. It's a dictionary, so you can get the keys
with open('/home/karimnazarovj/Rabbit/knowledge/mysql/system_view.json', 'r') as f:
    system_view_origin = json.load(f)
    all_knobs = list(system_view_origin.keys())

In [6]:
print('Entities shape:', entities_df.shape)
print('Relationships shape:', relationships_df.shape)

Entities shape: (80, 7)
Relationships shape: (51, 10)


In [7]:
entities_df[['name', 'type', 'description' ]].head(20)

,name,type,description
0,MYSQL,ORGANIZATION,MySQL is an open-source relational database ma...
1,MEMORY STORAGE ENGINE,ORGANIZATION,The MEMORY storage engine is used in MySQL for...
2,TEMPTABLE STORAGE ENGINE,ORGANIZATION,The TempTable storage engine is a component of...
3,INNODB,ORGANIZATION,InnoDB is a storage engine for MySQL renowned ...
4,TMP_TABLE_SIZE,EVENT,Defines the maximum size of internal in-memory...
5,MAX_HEAP_TABLE_SIZE,EVENT,Sets the maximum size to which user-created ME...
6,QUERY_PREALLOC_SIZE,EVENT,A deprecated MySQL variable as of version 8.0....
7,SORT_BUFFER_SIZE,EVENT,Defines the buffer size allocated for sorting ...
8,INNODB_BUFFER_POOL_SIZE,EVENT,Specifies the size of the buffer pool in InnoD...
9,INNODB_MAX_DIRTY_PAGES_PCT_LWM,EVENT,Defines a low water mark for dirty pages in In...


In [8]:
## Count how many entities in the graph are not knobs
all_knobs_lower = {k.lower() for k in all_knobs}

# get a list of all entitity names from dataframe
entity_names = entities_df['name'].tolist()
non_knob_entities = []
for entity_name in entity_names:
    # convert to lowercase and if more than one word, add '_' between words, then check if it's in all_knobs_lower
    entity_name_processed = entity_name.lower()
    if entity_name_processed not in all_knobs_lower:
        non_knob_entities.append(entity_name)


print('Total entities:', len(entity_names))
print('Non-knobs:', len(non_knob_entities))

Total entities: 80
Non-knobs: 22


In [9]:
# print some examples of non-knob entities 
entities_df[['name', 'type', 'description']].head(10)

,name,type,description
0,MYSQL,ORGANIZATION,MySQL is an open-source relational database ma...
1,MEMORY STORAGE ENGINE,ORGANIZATION,The MEMORY storage engine is used in MySQL for...
2,TEMPTABLE STORAGE ENGINE,ORGANIZATION,The TempTable storage engine is a component of...
3,INNODB,ORGANIZATION,InnoDB is a storage engine for MySQL renowned ...
4,TMP_TABLE_SIZE,EVENT,Defines the maximum size of internal in-memory...
5,MAX_HEAP_TABLE_SIZE,EVENT,Sets the maximum size to which user-created ME...
6,QUERY_PREALLOC_SIZE,EVENT,A deprecated MySQL variable as of version 8.0....
7,SORT_BUFFER_SIZE,EVENT,Defines the buffer size allocated for sorting ...
8,INNODB_BUFFER_POOL_SIZE,EVENT,Specifies the size of the buffer pool in InnoD...
9,INNODB_MAX_DIRTY_PAGES_PCT_LWM,EVENT,Defines a low water mark for dirty pages in In...


In [10]:
# show all non-knob entities
entities_df[entities_df['name'].isin(non_knob_entities)][['name', 'type', 'description']]

,name,type,description
0,MYSQL,ORGANIZATION,MySQL is an open-source relational database ma...
1,MEMORY STORAGE ENGINE,ORGANIZATION,The MEMORY storage engine is used in MySQL for...
2,TEMPTABLE STORAGE ENGINE,ORGANIZATION,The TempTable storage engine is a component of...
3,INNODB,ORGANIZATION,InnoDB is a storage engine for MySQL renowned ...
15,MYSQL 8.0.29,EVENT,A version of MySQL where the query_prealloc_si...
16,MYSQL 8.0.28,,
17,DDL OPERATION,EVENT,DDL (Data Definition Language) operations invo...
18,DML OPERATION,EVENT,DML (Data Manipulation Language) operations in...
19,ALTER TABLE OPERATION,EVENT,An ALTER TABLE operation is a specific type of...
39,MYSQL SERVER,ORGANIZATION,MySQL Server is a widely used open-source rela...


In [11]:
relationships_df[['source', 'target', 'weight', 'description']].head()

,source,target,weight,description
0,MYSQL,MEMORY STORAGE ENGINE,8.0,The MEMORY storage engine is used in MySQL for...
1,MYSQL,TEMPTABLE STORAGE ENGINE,8.0,The TempTable storage engine is used in MySQL ...
2,MYSQL,INNODB,10.0,InnoDB is a storage engine used within MySQL t...
3,MEMORY STORAGE ENGINE,TMP_TABLE_SIZE,7.0,TMP_TABLE_SIZE defines the maximum size of int...
4,MEMORY STORAGE ENGINE,MAX_HEAP_TABLE_SIZE,7.0,MAX_HEAP_TABLE_SIZE sets the maximum size to w...


In [12]:
# print how many total relationships
print('Total relationships:', relationships_df.shape[0])
# print how many relationships where source and target are both knobs
knob_relationships = relationships_df[
    relationships_df['source'].str.lower().isin(all_knobs_lower) &
    relationships_df['target'].str.lower().isin(all_knobs_lower)
]
print('Knob relationships:', knob_relationships.shape[0])

Total relationships: 51
Knob relationships: 2


In [13]:
# print some examples of knob relationships
knob_relationships[['source', 'target', 'weight', 'description']].head()

# print the entire description text
pd.set_option('display.max_colwidth', None)
knob_relationships[['source', 'target', 'weight', 'description']].head()

,source,target,weight,description
49,BINLOG_GROUP_COMMIT_SYNC_DELAY,BINLOG_GROUP_COMMIT_SYNC_NO_DELAY_COUNT,7.0,BINLOG_GROUP_COMMIT_SYNC_NO_DELAY_COUNT is dependent on the delay set by BINLOG_GROUP_COMMIT_SYNC_DELAY
50,INNODB_LOG_SPIN_CPU_ABS_LWM,INNODB_LOG_SPIN_CPU_PCT_HWM,1.0,Both variables define CPU usage thresholds for user threads spinning while waiting for flushed redo


In [14]:
# print some examples of relationships
relationships_df[['source', 'target', 'weight', 'description']]

,source,target,weight,description
0,MYSQL,MEMORY STORAGE ENGINE,8.0,The MEMORY storage engine is used in MySQL for creating internal in-memory temporary tables
1,MYSQL,TEMPTABLE STORAGE ENGINE,8.0,The TempTable storage engine is used in MySQL for creating internal in-memory temporary tables as of MySQL 8.0.28
2,MYSQL,INNODB,10.0,"InnoDB is a storage engine used within MySQL to manage database operations. It provides transaction-safe tables, ensuring that database transactions are processed reliably and securely."
3,MEMORY STORAGE ENGINE,TMP_TABLE_SIZE,7.0,TMP_TABLE_SIZE defines the maximum size of internal in-memory temporary tables created by the MEMORY storage engine
4,MEMORY STORAGE ENGINE,MAX_HEAP_TABLE_SIZE,7.0,MAX_HEAP_TABLE_SIZE sets the maximum size to which user-created MEMORY tables are permitted to grow
5,TEMPTABLE STORAGE ENGINE,TMP_TABLE_SIZE,7.0,TMP_TABLE_SIZE defines the maximum size of internal in-memory temporary tables created by the TempTable storage engine
6,TEMPTABLE STORAGE ENGINE,MYSQL 8.0.28,7.0,The TempTable storage engine was introduced in MySQL 8.0.28
7,INNODB,INNODB_BUFFER_POOL_SIZE,9.0,INNODB_BUFFER_POOL_SIZE specifies the size of the buffer pool in InnoDB
8,INNODB,INNODB_MAX_DIRTY_PAGES_PCT_LWM,8.0,INNODB_MAX_DIRTY_PAGES_PCT_LWM defines a low water mark for dirty pages in InnoDB
9,INNODB,INNODB_PURGE_THREADS,8.0,INNODB_PURGE_THREADS specifies the number of background threads for the InnoDB purge operation


## Entity extraction prompts

In [15]:
import json
from pathlib import Path

cache_dir = Path("/home/karimnazarovj/Rabbit/knowledge/postgres/graph/cache/entity_extraction")

def file_birth_time(path: Path):
    stat = path.stat()
    # On Linux, st_birthtime may not exist, so fall back to mtime.
    return getattr(stat, "st_birthtime", stat.st_mtime)

def extract_real_data_text(prompt_input: str) -> str:
    marker = "-Real Data-\n######################\nEntity_types:"
    start = prompt_input.find(marker)
    if start == -1:
        return ""

    text_marker = "\nText:"
    text_start = prompt_input.find(text_marker, start)
    if text_start == -1:
        return ""

    return prompt_input[text_start + len(text_marker):].strip()

rows = []

for path in sorted(cache_dir.iterdir(), key=file_birth_time):
    if not path.is_file():
        continue

    try:
        obj = json.loads(path.read_text())
    except Exception:
        continue

    rows.append({
        "file": path.name,
        "created": file_birth_time(path),
        "result": obj.get("result", "").strip(),
        "real_data_text": extract_real_data_text(obj.get("input", "")),
    })

# If you want the whole thing as a dataframe:
import pandas as pd
df = pd.DataFrame(rows)

# view
result = df[["file", "created", "real_data_text", "result"]]

# ToDo: Save this as a json array to output/entity_extraction_results.json 
result.to_json("/home/karimnazarovj/Rabbit/hallucination_discovery/output/entity_extraction_results.json", orient="records", indent=2)